In [43]:
import pandas as pd
from eop_data_common import utils


wpc_data = pd.read_csv('/data/eop/compiled_country_data/wdl_pov_clock_oct_2024.csv')
wpc_data = wpc_data[wpc_data.year == 2023]

aux_data = utils.get_latest_aux_data().drop(columns='country')

merged = wpc_data.merge(aux_data, left_on='ccode', right_on='country_code', how='left')

excluded_codes = [
    'HKG', # Hong Kong
    'KSV', # Kosovo
    'MAC', # Macao
    'PRI', # Puerto Rico
    'TKM', # Turkmenistan
    'TWN', # Taiwan
    'VIR', # US Virgin Islands
]

merged = merged[~merged.ccode.isin(excluded_codes)]
per_country_max_rate = 0.014
merged['rate_reduction'] = (merged['hcr_pov'] - per_country_max_rate).clip(lower=0)

assert merged[merged.rate_reduction > 0].total_population_2023.isna().sum() == 0
merged.fillna({'total_population_2023': 0}, inplace=True)


merged['headcount_reduction'] = merged.rate_reduction * merged.total_population_2023
num_lifted_out = merged.headcount_reduction.sum()
num_countries_affected = len(merged[merged.rate_reduction > 0])

print(int(round(num_lifted_out, 0)))
print(num_countries_affected)

532800769
85


In [9]:
with pd.option_context('display.max_rows', 200):
    display(wpc_data)

,country,ccode,year,pov_line,hc_pov,hcr_pov,pgi
23,Aruba,ABW,2023,2.15,375,0.003516,3.487252e-04
54,Afghanistan,AFG,2023,2.15,15932842,0.375984,1.393371e-01
85,Angola,AGO,2023,2.15,11015518,0.314115,1.183071e-01
116,Albania,ALB,2023,2.15,17814,0.006281,5.363669e-04
147,United Arab Emirates,ARE,2023,2.15,35828,0.003371,2.707946e-04
178,Argentina,ARG,2023,2.15,498444,0.010699,2.018807e-03
209,Armenia,ARM,2023,2.15,36132,0.012645,3.328868e-04
240,Antigua & Barbuda,ATG,2023,2.15,656,0.006260,3.957754e-04
271,Australia,AUS,2023,2.15,88139,0.003401,1.103371e-02
302,Austria,AUT,2023,2.15,41676,0.004708,2.017006e-02
